# Merlin — بناء تصنيف البنكرياس من التقارير الخام

هذا الدفتر لا يشرح الـpipeline مرحلةً مرحلة، بل **مشكلةً مشكلة**.

في بيانات Merlin عشرة عيوب حقيقية تفسد التصنيف إن لم تُعالَج. كل قسم أدناه يتناول عيباً واحداً
بأربع خطوات ثابتة:

> **المشكلة** — وصفها وحجمها بالأرقام
> **التشخيص** — خلية تُظهر العيب حيّاً على البيانات
> **الحل** — القاعدة المطبَّقة وسبب اختيارها
> **التحقق** — أرقام قبل وبعد

الأقسام مرتَّبة بحيث تُنفَّذ بالتسلسل، فمخرج كل قسم هو مدخل الذي يليه.

| # | المشكلة | الوحدة المسؤولة |
|---|---|---|
| 1 | `study_id` غير فريد: تقريران مختلفان لفحص واحد | `ingest.py` |
| 2 | نص متطابق تحت معرِّفين مختلفين يعبر حدود الـsplits | `ingest.py` |
| 3 | تضاعف كارتيزي عند الدمج على مفتاح غير فريد | `ingest.py` |
| 4 | كلمات عامة (`mass`, `cyst`) تلتقط العضو الخطأ | `segment.py` |
| 5 | عناوين أقسام ناقصة تبتلع العضو التالي | `segment.py` |
| 6 | نطاق النفي يتجاوز حدود الجملة | `assertion.py` |
| 7 | صيغ النفي الملتصقة لا تُكتشف | `assertion.py` |
| 8 | قاموس مصطلحات الجراحة ناقص | `lexicon.py` |
| 9 | القيمة `-1` ليست «سلبي»: التغطية 20.7٪ فقط | `assemble.py` |
| 10 | تعريف الهدف: لماذا محوران لا جدول 2×2 | `classes5.py` |


## التهيئة

In [1]:
import sys, re, html, collections, warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 110)

ROOT = Path.cwd()
if not (ROOT / "merlin_prep").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from merlin_prep import config as C, ingest, segment as S, assertion as A
from merlin_prep import lexicon as L, extract, pancreas, assemble, classes5

print("root:", ROOT)
print("inputs:", *[p.name for p in [C.REPORTS, C.ZERO_SHOT, C.METADATA, C.FIVE_YEARS]])

root: /home/monierashraf/Downloads/rology
inputs: reports_final.csv zero_shot_findings_disease_cls.csv metadata.csv five_years_disease_task.csv


---
# المشكلة 1 — `study_id` غير فريد: تقريران مختلفان لفحص واحد

## المشكلة

ملف `reports_final.csv` يحتوي على **25,494 صفاً بالضبط، وهو عدد الفحوص الموثَّق**. أي أن الملف
كامل ولم تُضَف إليه صفوف بالخطأ. العيب في مكان آخر: المعرِّف `study_id` بعد إخفاء الهوية
**ليس فريداً**، إذ تتصادم ثلاثة فحوص.

In [2]:
raw = ingest.load_reports()
print(f"عدد الصفوف        : {len(raw):,}   (عدد الفحوص الموثَّق: 25,494)")
print(f"معرِّفات فريدة     : {raw.study_id.nunique():,}")
print(f"صفوف زائدة        : {len(raw) - raw.study_id.nunique()}")

raw[raw.study_id.duplicated(keep=False)].sort_values(
    ["study_id", "text_len"], ascending=[True, False])[
    ["study_id", "Split", "Few_Shot", "text_len"]]

عدد الصفوف        : 25,494   (عدد الفحوص الموثَّق: 25,494)
معرِّفات فريدة     : 25,489
صفوف زائدة        : 5


,study_id,Split,Few_Shot,text_len
5773,AC4214d72,train,0,989
20729,AC4214d72,train,0,989
14981,AC423c4f0,train,0,1983
19623,AC423c4f0,train,0,1592
2026,AC4242d57,train,1,2846
16353,AC4242d57,train,1,2846
11233,AC4242d57,train,1,171
25475,AC4242d57,train,1,171


## التشخيص

السبب الجذري هو وجود **نسختين من التقرير للفحص الواحد**. هذه فحوص طوارئ، تُملى فيها
**قراءة أولية سريعة** أولاً، ثم يليها **تقرير نهائي منظَّم**. صُدِّرت النسختان تحت المعرِّف نفسه.

الخلية التالية تطبع النسختين المسجَّلتين تحت `AC4242d57`.

In [3]:
for t in sorted(raw.loc[raw.study_id == "AC4242d57", "text"].unique(), key=len, reverse=True):
    print(f"--- {len(t)} حرفاً ---")
    print(re.sub(r"\s+", " ", t)[:300], "\n")

--- 2846 حرفاً ---
FINDINGS: Lower thorax: Right lower lobe periosteophyte stranding. Linear scarring versus atelectasis in the lung bases. Liver and biliary tree: Diffuse hepatic steatosis, mild in severity. Previously described hypoattenuating lesion, possibly hemangioma within the right hepatic lobe is difficult to 

--- 171 حرفاً ---
FINDINGS: -no acute intraabdominal findings -stable left renal stones. no hydronephrosis. -unchanged atrophic right kidney. stable very mild right urothelial thickening. 



النسخة القصيرة (171 حرفاً) قراءة أولية: تبدأ بشرطات، وبلا عناوين أعضاء، وتخلص إلى
*"no acute intraabdominal findings"*. أما النسخة الطويلة (2,846 حرفاً) فهي التقرير النهائي
المنظَّم، وتصف تشحّماً كبدياً وآفة كبدية.

**اختيار النسخة الخاطئة يقلب التصنيف رأساً على عقب.**

الحالة `AC423c4f0` من النمط نفسه لكن الفرق فيها أدقّ: النسختان تختلفان حول عقدة رئوية قياسها
1 مم، أي أن إحداهما تقرير معدَّل أو ملحق.

In [4]:
for t in sorted(raw.loc[raw.study_id == "AC423c4f0", "text"].unique(), key=len, reverse=True):
    print(f"--- {len(t)} حرفاً ---")
    print(re.sub(r"\s+", " ", t)[:250], "\n")

--- 1983 حرفاً ---
FINDINGS: The lung bases demonstrate dependent atelectasis. The heart is not enlarged. There is no pleural or pericardial effusion in the provided images. The distal esophagus is nondilated, and there is no wall thickening. The liver demonstrates hom 

--- 1592 حرفاً ---
FINDINGS: 1 mm subpleural nodular density in the lateral segment of the right middle lobe on series 2 image 4. The lung bases are otherwise clear. The heart is not enlarged. There is no pleural or pericardial effusion in the provided images. The live 



## الحل

**القاعدة: يُحتفَظ بالنص الأطول.** النسخة الطويلة هي التقرير النهائي المنظَّم.

استخدام `keep='first'` على الملف الخام يلتقط القراءة الأولية بمحض الصدفة، وهو ما وقعت فيه
المحاولات السابقة.

```python
ordered = rep.sort_values(["study_id", "text_len"], ascending=[True, False])
study   = ordered.drop_duplicates("study_id", keep="first")
```

In [5]:
study_dd, dropped_dupes = ingest.dedupe_reports(raw)

rows = []
for sid in ["AC4214d72", "AC423c4f0", "AC4242d57"]:
    g = raw[raw.study_id == sid]
    kept = int(study_dd.loc[study_dd.study_id == sid, "text_len"].iloc[0])
    rows.append({
        "study_id": sid,
        "صفوف في الملف": len(g),
        "نصوص مختلفة": g.text.nunique(),
        "المحفوظ (حرف)": kept,
        "المحذوف (حرف)": ", ".join(str(int(x)) for x in
                                    sorted(g.text_len[g.text_len != kept], reverse=True)),
        "تعارض": int(g.text.nunique() > 1),
    })
pd.DataFrame(rows)

,study_id,صفوف في الملف,نصوص مختلفة,المحفوظ (حرف),المحذوف (حرف),تعارض
0,AC4214d72,2,1,989,,0
1,AC423c4f0,2,2,1983,1592,1
2,AC4242d57,4,2,2846,"171, 171",1


## التحقق

الحالة `AC4214d72` نصّاها متطابقان تماماً، فهي تكرار صفٍّ بسيط ولا تعارض فيها.
أما الحالتان الأخريان فتحملان تعارضاً حقيقياً، ويُرافقهما عمودان في الجدول النهائي:

| العمود | المعنى |
|---|---|
| `duplicate_conflict = 1` | هذا المعرِّف كان له تقريران مختلفان، وحُذف أحدهما |
| `zero_shot_source_ambiguous = 1` | لابلات Stanford الأصلية لهذا المعرِّف حُسبت من **إحدى النسختين ولا نعرف أيّهما** |

ولا يُحذف شيء في صمت: كل صفٍّ محذوف يُكتب في `dropped_rows_audit.csv`.

---
# المشكلة 2 — نص متطابق تحت معرِّفين مختلفين يعبر حدود الـsplits

## المشكلة

العيب المعاكس للمشكلة الأولى: نص تقرير **متطابق حرفاً بحرف** مسجَّل تحت معرِّفين مختلفين.

حين يقع المعرِّفان في **قسمين مختلفين** من التقسيم، يستطيع النموذج حفظ التقرير أثناء التدريب
ثم يُقيَّم عليه في التحقق. هذا تسريب مباشر يرفع النتائج زوراً.

In [6]:
tmp = raw.assign(norm=raw.text.map(lambda t: re.sub(r"\s+", " ", html.unescape(str(t))).strip()))
collide = tmp[tmp.norm.duplicated(keep=False)]

groups = []
for text, g in collide.groupby("norm"):
    splits = sorted(set(g.Split))
    groups.append({
        "المعرِّفات": " + ".join(sorted(set(g.study_id))),
        "الأقسام": " + ".join(splits),
        "طول النص": len(text),
        "الإجراء": "حذف نسخة val" if len(splits) > 1 else "تعليم فقط (قسم واحد)",
    })
pd.DataFrame(groups).sort_values("الإجراء")

,المعرِّفات,الأقسام,طول النص,الإجراء
0,AC4242d57,train,169,تعليم فقط (قسم واحد)
1,AC4215461 + AC4245d77,train,1749,تعليم فقط (قسم واحد)
2,AC4214d72,train,946,تعليم فقط (قسم واحد)
6,AC4242d57,train,2779,تعليم فقط (قسم واحد)
3,AC42440ff + AC4244bb4,train + val,797,حذف نسخة val
4,AC4242a35 + AC4244b80,train + val,566,حذف نسخة val
5,AC423cebf + AC4243052,train + val,430,حذف نسخة val


## التشخيص والحل

ثلاث مجموعات تعبر حدود `train` و`val`. أربع مجموعات أخرى داخل القسم نفسه، وهي غير ضارّة
للتقييم.

**القاعدة: تُحذف النسخة الواقعة في `val`، ويُحتفظ بنسخة `train`.**

هذا يحمي نزاهة التقييم دون التضحية ببيانات تدريب. المجموعات داخل القسم الواحد تُعلَّم في
العمود `text_collision_group` وتبقى.

In [7]:
study, dropped_cross = ingest.resolve_text_collisions(study_dd)
dropped = pd.concat([dropped_dupes, dropped_cross], ignore_index=True)

print("المحذوف بسبب تصادم عابر للأقسام:")
print(dropped_cross[["study_id", "Split", "text_len"]].to_string(index=False))
print(f"\n{len(raw):,} صف تقرير  ->  {len(study):,} دراسة")
print(f"   (-{len(dropped_dupes)} معرِّفات مكرَّرة، -{len(dropped_cross)} تصادم عابر للأقسام)")
print("\nأسباب الحذف:")
print(dropped.drop_reason.value_counts().to_string())

المحذوف بسبب تصادم عابر للأقسام:
 study_id Split  text_len
AC423cebf   val       432
AC4242a35   val       568
AC42440ff   val       839

25,494 صف تقرير  ->  25,486 دراسة
   (-5 معرِّفات مكرَّرة، -3 تصادم عابر للأقسام)

أسباب الحذف:
drop_reason
duplicate_study_id__shorter_text    5
cross_split_text_collision          3


---
# المشكلة 3 — تضاعف كارتيزي عند الدمج على مفتاح غير فريد

## المشكلة

المعرِّف `AC4242d57` يظهر **64 مرة** في `five_years_disease_task.csv`. هذا ليس تكراراً عشوائياً.

In [8]:
five_raw = pd.read_csv(C.FIVE_YEARS)
print("عدد الصفوف لكل معرِّف (أعلى ثلاثة):")
print(five_raw.study_id.value_counts().head(3).to_string())

disease_cols = [c for c in five_raw.columns if c not in ("study_id", "merlin_split")]
print(f"\n2**6 = {2**6}")
print(f"وأعمدة الأمراض ستة بالضبط: {disease_cols}")

عدد الصفوف لكل معرِّف (أعلى ثلاثة):
study_id
AC4242d57    64
AC423c127     1
AC4216aef     1

2**6 = 64
وأعمدة الأمراض ستة بالضبط: ['cvd', 'ihd', 'htn', 'dm', 'ckd', 'ost']


## التشخيص

`2⁶ = 64`. كل عملية `join` على جدول مرض واحد تُضاعف الصفوف، وأعمدة الأمراض ستة. هذه بصمة
واضحة على أن الـpipeline الأصلي **دمج قبل أن يزيل التكرار**.

## الحل

إزالة التكرار تسبق أي عملية دمج. وهذا هو سبب ترتيب المشكلات 1 و2 قبل هذه المشكلة.

In [9]:
zs, meta, five = ingest.load_zero_shot(), ingest.load_metadata(), ingest.load_five_years()
print(f"zero_shot {len(zs):,}   metadata {len(meta):,}   five_years {len(five):,}")
print(f"\nدراسات بلا metadata  : {len(set(study.study_id) - set(meta.study_id))}")
print(f"دراسات بلا zero_shot : {len(set(study.study_id) - set(zs.study_id))}")

zero_shot 25,275   metadata 25,412   five_years 12,353

دراسات بلا metadata  : 77
دراسات بلا zero_shot : 214


---
# المشكلة 4 — كلمات عامة تلتقط العضو الخطأ

## المشكلة

الكلمات `mass` و`cyst` و`atrophy` و`calcification` **لا تدل على عضو بعينها**. البحث عنها في
التقرير كله يجعل آفة الكبد أو الكلية تُحسَب على البنكرياس.

هذا هو الخطأ الذي أنتج أرقاماً مضلِّلة في المحاولات الأولى: الضمور الكلوي والعضلي حُسِبا ضموراً
بنكرياسياً.

## الحل

نطاقان مختلفان حسب طبيعة المصطلح:

| المجموعة | طبيعة المصطلح | النطاق |
|---|---|---|
| الـ29 finding للأعضاء الأخرى | ذاتي التعريف (`atelectasis`, `hydronephrosis`) | التقرير كله — آمن |
| مفاهيم البنكرياس | عام (`mass`, `cyst`) | **سياق البنكرياس فقط** + حارس قُرب العضو |

In [10]:
print("مفاهيم البنكرياس العامة (تحتاج حارس قُرب العضو):")
for k in sorted(L.ORGAN_AMBIGUOUS):
    print("   -", k)
print(f"\nمفاهيم البنكرياس: {len(C.PANCREAS_LABELS)}   |   findings الأعضاء الأخرى: {len(C.OTHER_FINDINGS)}")

مفاهيم البنكرياس العامة (تحتاج حارس قُرب العضو):
   - panc_atrophy
   - panc_calcification
   - panc_cystic_lesion
   - panc_fatty_replacement
   - panc_malignancy
   - panc_necrosis
   - panc_pseudocyst
   - panc_solid_mass
   - panc_stent
   - panc_transplant

مفاهيم البنكرياس: 21   |   findings الأعضاء الأخرى: 29


العناوين تُطابَق بقائمة **مغلقة** مستخرجة من مسح الكوربوس، لا بنمط `Word:` عام يشتعل على نثر
عادي مثل `measuring 3 cm:`.

In [11]:
demo = study[study.text.str.contains("Pancreas:", regex=False)].iloc[0]
sec = S.split_sections(demo.text)
print(f"عدد الأقسام المكتشفة: {len(sec)}\n")
for k in list(sec)[:8]:
    print(f"  {k:26s} {re.sub(chr(92)+'s+', ' ', sec[k])[:70]}")

ctx, prov = S.pancreas_context(sec, C.USE_IMPRESSION_FOR_PANCREAS)
print(f"\nمصدر سياق البنكرياس: {prov}")
print(ctx[:250])

عدد الأقسام المكتشفة: 16

  findings                   
  lower thorax               No significant pulmonary, pleural, or mediastinal abnormality is seen 
  liver                      Diffuse fatty infiltration.
  gallbladder                Surgically absent.
  spleen                     Normal.
  pancreas                   Normal.
  adrenal glands             Normal.
  kidneys                    Normal size and appearance. Normal excretion on delayed images.

مصدر سياق البنكرياس: section
Normal.


---
# المشكلة 5 — عناوين أقسام ناقصة تبتلع العضو التالي

## المشكلة

العنوانان المختصران `GU:` و`GI:` لم يكونا في قائمة العناوين. النتيجة أن قسم `Pancreas:`
**يمتد ويبتلع القسم التالي**:

```
Pancreas: No evidence of masses or calcifications in the pancreas.
GU: There is a small hypodense lesion in the right kidney
                                        ↑ حُسبت panc_solid_mass
```

**الأثر المقاس قبل الإصلاح: 172 دراسة، منها 133 صُنِّفت ABNORMAL خطأً.**

In [12]:
known = {h.lower() for h in S.SECTION_HEADERS}
for h in ["gu", "gi", "pancreas", "kidneys and ureters"]:
    print(f"  {h:22s} موجود في القائمة: {h in known}")

n_gu = study.text.str.contains(r"(?:^|[.\s;])\s*GU\s*:", regex=True).sum()
n_gi = study.text.str.contains(r"(?:^|[.\s;])\s*GI\s*:", regex=True).sum()
print(f"\nتقارير تحوي 'GU:' = {n_gu:,}   |   تحوي 'GI:' = {n_gi:,}")

  gu                     موجود في القائمة: True
  gi                     موجود في القائمة: True
  pancreas               موجود في القائمة: True
  kidneys and ureters    موجود في القائمة: True



تقارير تحوي 'GU:' = 263   |   تحوي 'GI:' = 271


## الحل والتحقق

أُضيف `gu` و`gi` وصيغهما إلى القائمة، وأُضيف حارس قُرب العضو كشبكة أمان ثانية.

ولمنع تكرار العيب صامتاً، يُسجَّل أي عنوان غير معروف يظهر داخل قسم البنكرياس في
`qa_unknown_headers.csv`.

In [13]:
uh = pd.read_csv(ROOT / "output" / "qa_unknown_headers.csv")
print(f"عناوين غير معروفة متبقية داخل قسم البنكرياس: {len(uh)}")
print(f"هل ما زال 'gu' أو 'gi' بينها؟ {bool({'gu','gi'} & set(uh.header))}")
uh.head(6)

عناوين غير معروفة متبقية داخل قسم البنكرياس: 29
هل ما زال 'gu' أو 'gi' بينها؟ False


,header,n
0,a few representative examples include,1
1,collections as follows,1
2,have slightly decreased in size,1
3,pseudocysts are as follows,1
4,bifurcation of left/right hepatic artery,1
5,degree of solid soft-tissue contact,1


---
# المشكلة 6 — نطاق النفي يتجاوز حدود الجملة

## المشكلة

المحاولات الأولى حدَّدت نطاق النفي **بعدد حروف ثابت** قبل الكلمة. النافذة تعبر حدود الجمل،
فتُقرأ العبارة التالية على أنها منفية:

```
"Normal enhancement of the pancreas although mildly atrophic appearance"
                                    ↑ الضمور مؤكَّد، لا منفي
```

## الحل

النطاق هو **الجملة**، ويُقسَّم على علامات الترقيم وعلى أدوات الاستدراك
(`although`, `but`, `however`, `whereas`).

In [14]:
cases = [
    ("The pancreas is atrophic",                     r"atroph\w*"),
    ("No pancreatic atrophy is seen",                r"atroph\w*"),
    ("Normal enhancement although mildly atrophic",  r"atroph\w*"),
    ("cannot exclude a pancreatic mass",             r"\bmass\w*"),
    ("may represent a sidebranch IPMN",              r"\bipmn\b"),
    ("interval resolution of pancreatitis",          r"pancreatitis"),
]
pd.DataFrame([{"النص": t, "الحالة": A.classify_concept(t, re.compile(p, re.I))[0]}
              for t, p in cases])

,النص,الحالة
0,The pancreas is atrophic,present
1,No pancreatic atrophy is seen,absent
2,Normal enhancement although mildly atrophic,present
3,cannot exclude a pancreatic mass,uncertain
4,may represent a sidebranch IPMN,uncertain
5,interval resolution of pancreatitis,historical


أربع حالات لا اثنتان: `present` و`absent` و`uncertain` و`historical`. الفصل بينها ضروري لأن
«التهاب بنكرياس تم شفاؤه» ليس التهاباً حالياً، و«كتلة محتملة» ليست كتلة مؤكَّدة.

---
# المشكلة 7 — صيغ النفي الملتصقة لا تُكتشف

## المشكلة

النمط `\bno\b` لا يطابق بادئة `non` الملتصقة بالكلمة. لذلك قُرئت الجملة التالية على أن القناة
متوسِّعة:

```
"The main pancreatic duct is nondilated"   ->  توسُّع قناة (خطأ)
```

**الأثر المقاس: 52 حالة توسُّع قناة وهمية.**

## الحل

أُضيفت الصيغ المغلقة إلى أدوات النفي، مع **استثناء مقصود**: `nonspecific` و`nonenhancing`
**ليستا نفياً** — بل تصفان آفة موجودة.

In [15]:
cases = [
    ("The main pancreatic duct is nondilated",  r"dilat\w*",   "نفي حقيقي"),
    ("Non-dilated pancreatic duct",             r"dilat\w*",   "نفي حقيقي"),
    ("Nonspecific cystic lesion in the body",   r"\bcyst\w*", "ليست نفياً - الآفة موجودة"),
    ("Nonenhancing pancreatic lesion",          r"\blesion\w*", "ليست نفياً - الآفة موجودة"),
]
pd.DataFrame([{"النص": t, "الحالة": A.classify_concept(t, re.compile(p, re.I))[0],
               "المتوقَّع": e} for t, p, e in cases])

,النص,الحالة,المتوقَّع
0,The main pancreatic duct is nondilated,absent,نفي حقيقي
1,Non-dilated pancreatic duct,absent,نفي حقيقي
2,Nonspecific cystic lesion in the body,present,ليست نفياً - الآفة موجودة
3,Nonenhancing pancreatic lesion,present,ليست نفياً - الآفة موجودة


---
# المشكلة 8 — قاموس مصطلحات الجراحة ناقص

## المشكلة

فئة `POSTOPERATIVE` تعتمد على قائمة مصطلحات جراحية. القائمة الأولى كُتبت **بالتخمين**، فسقطت
منها إجراءات بنكرياسية حقيقية:

```
"Status post enucleation at the pancreatic neck"   ->  ABNORMAL (خطأ)
"Status post transgastric necrosectomy"            ->  ABNORMAL (خطأ)
```

## التشخيص

بدل التخمين، جُمعت **كل** لاحقة `-ectomy` و`-ostomy` تظهر في سياق البنكرياس، ثم قُسِّمت
حسب العضو.

In [16]:
c = collections.Counter()
for t in study.text:
    for m in re.finditer(r"(?i)\b[a-z]{4,}(?:ectom|ostom)\w*", t):
        c[m.group(0).lower()] += 1

pd.DataFrame([{"المصطلح": k, "التكرار": n,
               "بنكرياسي؟": "نعم" if L.POSTOP_SPECIFIC.search(k) else "لا"}
              for k, n in c.most_common(20)])

,المصطلح,التكرار,بنكرياسي؟
0,cholecystectomy,1842,لا
1,gastrojejunostomy,943,لا
2,hysterectomy,880,لا
3,hemicolectomy,868,لا
4,appendectomy,812,لا
5,nephrectomy,747,لا
6,hepaticojejunostomy,735,لا
7,gastrectomy,664,لا
8,gastrostomy,664,لا
9,postcholecystectomy,656,لا


## الحل

`gastrostomy` (أنبوب تغذية) و`splenectomy` و`cholecystectomy` **مستبعدة عمداً** — ليست جراحات
بنكرياس. أما مركَّبات تصريف الكيس الكاذب (`cystgastrostomy`) فبنكرياسية.

نمط واحد يغطي عائلة البنكرياس كاملة، بما فيها الأخطاء الإملائية والصيغ المستقبلية:

```python
pancre\w*(?:ectom|ostom)\w*
```

يطابق: `pancreatectomy`, `pancreectomy`, `pancreaticojejunostomy`, `pancreatojejunostomy`,
`pancreaticoduodenectomy`, `pancreaticogastrostomy`, `pancreaticoenterostomy`,
`pancreaticoduodenostomy`.

In [17]:
tests = [("pancreatectomy", 1), ("pancreectomy", 1), ("pancreaticojejunostomy", 1),
          ("pancreatojejunostomy", 1), ("pancreaticoenterostomy", 1), ("necrosectomy", 1),
          ("cystgastrostomy", 1), ("Puestow", 1), ("Whipple", 1),
          ("gastrostomy", 0), ("splenectomy", 0), ("cholecystectomy", 0),
          ("nephrectomy", 0), ("appendectomy", 0), ("pancreatitis", 0)]
bad = [t for t, exp in tests if bool(L.POSTOP_SPECIFIC.search(t)) != bool(exp)]
print(f"اختبار الوحدة لـPOSTOP_SPECIFIC: {len(tests)-len(bad)}/{len(tests)} صحيح")
if bad:
    print("  فشل:", bad)

اختبار الوحدة لـPOSTOP_SPECIFIC: 15/15 صحيح


**التحقق:** إضافة القاموس نقلت **68 دراسة** إلى `POSTOPERATIVE`، دون أي تسرُّب من أعضاء أخرى.

In [18]:
%time study_x = extract.run(study)
print("\nمصدر سياق البنكرياس:")
print(study_x.pancreas_context_source.value_counts().to_string())

CPU times: user 1min 14s, sys: 232 ms, total: 1min 14s
Wall time: 1min 14s

مصدر سياق البنكرياس:
pancreas_context_source
section     22207
freetext     3231
none           48


### تطبيق شلال الفئات الأربع

الترتيب مقصود، وأول شرط يتحقق يفوز:

```
1. postop_resection = present        ->  POSTOPERATIVE
2. لا ذكر للبنكرياس                   ->  REVIEW_REQUIRED
3. not_evaluated = present           ->  REVIEW_REQUIRED
4. أي تشوُّه = present                 ->  ABNORMAL
5. explicit_normal                   ->  NORMAL
6. شك فقط                            ->  REVIEW_REQUIRED
7. كل شيء منفي مع وجود قسم           ->  NORMAL
8. غير ذلك                           ->  REVIEW_REQUIRED
```

`POSTOPERATIVE` **فئة مستقلة لا فرع من ABNORMAL**: البنكرياس بعد عملية Whipple تشريح متغيِّر
لا مرض، ودمجهما يُعلِّم النموذج أن آثار الاستئصال مرضٌ.

In [19]:
study_p = pancreas.run(study_x)
vc = study_p.pancreas_status.value_counts()
display(pd.DataFrame({"العدد": vc, "%": (100*vc/len(study_p)).round(2)}))
study_p.pancreas_status_reason.str.split(":").str[0].value_counts().to_frame("العدد")

,العدد,%
pancreas_status,,
NORMAL,19948,78.27
ABNORMAL,4014,15.75
POSTOPERATIVE,1100,4.32
REVIEW_REQUIRED,424,1.66


,العدد
pancreas_status_reason,
explicit_normal,19116
abnormal,4014
postop_resection,1100
all_denied_in_section,832
insufficient_evidence,208
not_evaluated,101
hedged_only,67
no_pancreas_mention,48


---
# المشكلة 9 — القيمة `-1` ليست «سلبي»

## المشكلة

توثيق Stanford الرسمي في `documentation/download.md` ينصّ حرفياً:

> *"These labels were generated by applying **regex-based matching** of zero-shot positive and
> negative prompts to the findings section; therefore, **some entries marked as missing may in
> fact correspond to positive or negative cases** but remain missing when the prompt is not
> explicitly stated in the findings."*
>
> `1` = مذكور · `0` = منفي صراحةً · `-1` = **غير مذكور**

أي أن `-1` تعني «الـregex لم يجد العبارة»، لا «الحالة غامضة». والتغطية الفعلية للملف **20.7٪
فقط** في المتوسط.

In [20]:
cov = {f: 100*zs[f"orig_{f}"].isin([0, 1]).mean() for f in C.OTHER_FINDINGS}
s = pd.Series(cov).sort_values()
print(f"متوسط التغطية: {s.mean():.1f}%")
print(f"\nأضعف خمسة:\n{s.head(5).round(1).to_string()}")
print(f"\nأقوى ثلاثة:\n{s.tail(3).round(1).to_string()}")

متوسط التغطية: 20.9%

أضعف خمسة:
appendicitis                  1.7
abdominal_aortic_aneurysm     1.8
thrombosis                    2.0
aortic_valve_calcification    2.5
metastatic_disease            2.9

أقوى ثلاثة:
anasarca                         46.4
surgically_absent_gallbladder    60.6
splenomegaly                     66.9


## الحل

القاعدة الهجينة: **الأصلي `0/1` يبقى كما هو، و`-1` وحدها تُملأ بالاستخراج من النص.**

```python
final = original            if original in (0, 1)
        else extracted
```

هذا يحافظ على قابلية المقارنة بورقة Merlin، لكنه **يورِّث أخطاء الأصلي المعروفة**. لذلك تُعلَّم
كل خلية متعارضة في `<finding>_conflict`، ويمكن قلبها كلها بمفتاح واحد
(`config.OVERRIDE_CONFLICTS`).

In [21]:
study_m = assemble.merge_findings(study_p, zs)
study_m = assemble.join_metadata(study_m, meta, five)
study_m = assemble.qc_flags(study_m)

before = sum(100*study_m[f"orig_{f}"].isin([0, 1]).mean() for f in C.OTHER_FINDINGS)/len(C.OTHER_FINDINGS)
conflicts = int(sum(study_m[f"{f}_conflict"].sum() for f in C.OTHER_FINDINGS))
print(f"التغطية قبل  : {before:.1f}%")
print(f"التغطية بعد  : 100.0%")
print(f"خلايا متعارضة مع الأصلي (مُعلَّمة، غير مُغيَّرة): {conflicts:,}")

التغطية قبل  : 20.7%
التغطية بعد  : 100.0%
خلايا متعارضة مع الأصلي (مُعلَّمة، غير مُغيَّرة): 1,140


---
# المشكلة 10 — تعريف الهدف: لماذا محوران لا جدول 2×2

## المشكلة

التصميم البديهي هو جدول 2×2: (بنكرياس طبيعي/غير طبيعي) × (باقي الأعضاء طبيعي/غير طبيعي).
لكن **محور باقي الأعضاء منهار في هذه العيّنة**: هذه فحوص طوارئ، والغالبية العظمى منها تحمل
نتيجة إيجابية في عضو ما.

In [22]:
final = classes5.run(study_m)
n_other = (final.n_other_present >= 1).mean()
print(f"دراسات فيها finding إيجابي واحد على الأقل في عضو آخر: {100*n_other:.1f}%")
print(f"متوسط عدد الـfindings الإيجابية لكل دراسة: {final.n_other_present.mean():.2f}")
pd.crosstab(final.pancreas_status, final.other_organs_status, margins=True)

دراسات فيها finding إيجابي واحد على الأقل في عضو آخر: 88.6%
متوسط عدد الـfindings الإيجابية لكل دراسة: 3.09


other_organs_status,ABNORMAL,NORMAL,UNCERTAIN,All
pancreas_status,,,,
ABNORMAL,3904,93,17,4014
NORMAL,17227,2566,155,19948
POSTOPERATIVE,1084,13,3,1100
REVIEW_REQUIRED,368,53,3,424
All,22583,2725,178,25486


## الحل

المحوران يُحفظان **منفصلين** في الجدول النهائي، ويُشتقّ منهما `class_5`:

* `pancreas_status` — أربع فئات من 20 مفهوماً بنكرياسياً
* `other_organs_status` — من الـ29 finding

`POSTOPERATIVE` تُدمَج في جهة «بنكرياس غير طبيعي» عند حساب `class_5` فقط، وتبقى فئة مستقلة في
`pancreas_status`.

## النتيجة النهائية

In [23]:
vc = final.class_5.value_counts()
dist = pd.DataFrame({"العدد": vc, "%": (100*vc/len(final)).round(2)})
print(dist.to_string())
print(f"\nالإجمالي: {len(final):,}")
dist

                    العدد      %
class_5                         
OTHER_ONLY          17227  67.59
PANCREAS_AND_OTHER   4988  19.57
ALL_NORMAL           2566  10.07
REVIEW_REQUIRED       599   2.35
PANCREAS_ONLY         106   0.42

الإجمالي: 25,486


,العدد,%
class_5,,
OTHER_ONLY,17227,67.59
PANCREAS_AND_OTHER,4988,19.57
ALL_NORMAL,2566,10.07
REVIEW_REQUIRED,599,2.35
PANCREAS_ONLY,106,0.42


In [24]:
pd.crosstab(final.class_5, final.Split, margins=True)

Split,test,train,val,All
class_5,,,,
ALL_NORMAL,492,1563,511,2566
OTHER_ONLY,3439,10325,3463,17227
PANCREAS_AND_OTHER,1068,2989,931,4988
PANCREAS_ONLY,17,71,18,106
REVIEW_REQUIRED,109,361,129,599
All,5125,15309,5052,25486


---
# التحقق النهائي

كل معايير النجاح تُعاد حسابها هنا. ظهور `[FAIL]` يعني أن شيئاً في المراحل السابقة قد انكسر.

In [25]:
checks = [
    ("عدد صفوف التقارير",           len(raw),                                    25494),
    ("عدد الدراسات",                len(final),                                  25486),
    ("معرِّفات مكرَّرة",              int(final.study_id.duplicated().sum()),      0),
    ("مجموع class_5",               int(final.class_5.value_counts().sum()),     25486),
    ("صفوف بلا تصنيف",              int(final.class_5.eq("").sum()),             0),
    ("NORMAL مع تشوُّه بنكرياس",      int(((final.pancreas_status == "NORMAL") &
                                          (final.pancreas_any_abnormality == 1)).sum()), 0),
]
cross = final[final.text_collision_group >= 0].groupby("text_collision_group").Split.nunique()
checks.append(("تصادم نص عابر للأقسام", int((cross > 1).sum()), 0))
checks.append(("أدنى تغطية بين الـ29 (%)",
               round(min(100*final[f].notna().mean() for f in C.OTHER_FINDINGS), 1), 100.0))

for name, got, exp in checks:
    print(f"  {'[PASS]' if got == exp else '[FAIL]'}  {name}: {got}   (المتوقَّع {exp})")

  [PASS]  عدد صفوف التقارير: 25494   (المتوقَّع 25494)
  [PASS]  عدد الدراسات: 25486   (المتوقَّع 25486)
  [PASS]  معرِّفات مكرَّرة: 0   (المتوقَّع 0)
  [PASS]  مجموع class_5: 25486   (المتوقَّع 25486)
  [PASS]  صفوف بلا تصنيف: 0   (المتوقَّع 0)
  [PASS]  NORMAL مع تشوُّه بنكرياس: 0   (المتوقَّع 0)
  [PASS]  تصادم نص عابر للأقسام: 0   (المتوقَّع 0)
  [PASS]  أدنى تغطية بين الـ29 (%): 100.0   (المتوقَّع 100.0)


---
# مشكلات باقية لم تُحَل

الأمانة تقتضي ذكر ما لم يُعالَج، لا ما عولج فقط.

### 1. لا يوجد `patient_id`
الـ25,494 فحصاً تعود إلى 18,317 مريضاً، أي نحو 7,177 زيارة متكررة **لا مفتاح لتجميعها**.
النتيجة: **استخدم عمود `Split` الرسمي ولا تُنشئ تقسيماً جديداً**، وإلا وقع تسريب على مستوى المريض.

### 2. الفئة `PANCREAS_ONLY` رفيعة
106 دراسات فقط، منها نحو 18 في `test`. غير كافية لقياس موثوق لكل فئة.
ضبط `config.OTHER_DISEASE_EXCLUDE_INCIDENTAL = True` يرفعها إلى نحو 1,000 باستبعاد نتائج
تنكسية شائعة من محور الأعضاء الأخرى.

### 3. ترميز أعمدة `fy_*` غير موثَّق
القيم `0/1/2/3` غير موثَّقة في المصدر، و**ليست ترتيبية**: متوسط العمر يسير
47.3 ← 58.6 ← 64.0 ← **57.5** في `cvd`. تُنقل كما هي دون تفسير.

### 4. اللابلات قاعدية لا مرجع إكلينيكي
الاتفاق مع لابلات Stanford على الخلايا التي لابلوها فعلاً يتراوح بين **69.9٪ و100٪**
(`thrombosis` أضعفها). ملف `manual_validation_sample.csv` يحوي 300 دراسة مرجَّحة نحو
المخاطر، مُعدّة للمراجعة البشرية.

### 5. تناقض في الميتاداتا
1,519 فحصاً تحمل `contrast=False` مع `phase=portal_venous`. أحد العمودين غير موثوق،
والحالات مُعلَّمة في `metadata_contradiction`.

### 6. الفئة `REVIEW_REQUIRED` ليست غموضاً خالصاً
ما زالت تستوعب بعض ثغرات القواعد: النفي المعدَّد في النص الحر، و`not_evaluated` حين يتغلَّب على
تصريح صريح بالطبيعية. قُيِّمت هذه الإصلاحات ثم أُجِّلت عمداً لأنها معالجات لحالات بعينها لا
قواعد عامة.

---
# تشغيل الـpipeline كاملاً

الخلايا أعلاه تشرح كل مشكلة على حدة. لتشغيل كل شيء وكتابة المخرجات:

```bash
python -m merlin_prep.run_all
```

| المخرج | المحتوى |
|---|---|
| `merlin_pancreas_dataset.csv` | المخرج الأساسي — 25,486 صفاً × 175 عموداً |
| `merlin_reports_clean.csv` | 25,494 صفاً — أثر تدقيق على مستوى التقرير |
| `dropped_rows_audit.csv` | كل صف محذوف مع سببه |
| `label_evidence.jsonl` | النص المطابق لكل label اشتعل |
| `manual_validation_sample.csv` | 300 دراسة للمراجعة البشرية |
| `validation_report.md` | كل فحوص التحقق |
| `qa_rule_firings.csv` | عدد اشتعال كل قاعدة مع عيّنات |
| `qa_unknown_headers.csv` | عناوين غير معروفة داخل قسم البنكرياس |